![Credit card being held in hand](credit_card.jpg)

Commercial banks receive _a lot_ of applications for credit cards. Many of them get rejected for many reasons, like high loan balances, low income levels, or too many inquiries on an individual's credit report, for example. Manually analyzing these applications is mundane, error-prone, and time-consuming (and time is money!). Luckily, this task can be automated with the power of machine learning and pretty much every commercial bank does so nowadays. In this workbook, you will build an automatic credit card approval predictor using machine learning techniques, just like real banks do.

### The Data

The data is a small subset of the Credit Card Approval dataset from the UCI Machine Learning Repository showing the credit card applications a bank receives. This dataset has been loaded as a `pandas` DataFrame called `cc_apps`. The last column in the dataset is the target value.

In [36]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV

# Load the dataset
cc_apps = pd.read_csv("cc_approvals.data", header=None) 
cc_apps.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,b,30.83,0.000,u,g,w,v,1.25,t,t,1,g,0,+
1,a,58.67,4.460,u,g,q,h,3.04,t,t,6,g,560,+
2,a,24.50,0.500,u,g,q,h,1.50,t,f,0,g,824,+
3,b,27.83,1.540,u,g,w,v,3.75,t,t,5,g,3,+
4,b,20.17,5.625,u,g,w,v,1.71,t,f,0,s,0,+


In [37]:
cc_apps = cc_apps.replace("?", np.nan)
X = cc_apps.iloc[:, :-1].copy()
y = cc_apps.iloc[:, -1]
num_cols = [1,2,7,10,12]
for col in num_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce")
for col in num_cols:
    X[col] = X[col].fillna(X[col].median())
for col in X.columns:
    if col not in num_cols:
        X[col] = X[col].fillna(X[col].mode()[0])
X = pd.get_dummies(X)
X.head()


,1,2,7,10,12,0_a,0_b,3_l,3_u,3_y,4_g,4_gg,4_p,5_aa,5_c,5_cc,5_d,5_e,5_ff,5_i,5_j,5_k,5_m,5_q,5_r,5_w,5_x,6_bb,6_dd,6_ff,6_h,6_j,6_n,6_o,6_v,6_z,8_f,8_t,9_f,9_t,11_g,11_p,11_s
0,30.83,0.000,1.25,1,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,1,1,0,0
1,58.67,4.460,3.04,6,560,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,1,1,0,0
2,24.50,0.500,1.50,0,824,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,1,0,1,0,0
3,27.83,1.540,3.75,5,3,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,1,1,0,0
4,20.17,5.625,1.71,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,1


In [38]:

X = pd.get_dummies(X)
X = X.apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median()).fillna(0)
X.columns = X.columns.astype(str)
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# Scale the data
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print("Step 2 completed successfully!")




Step 2 completed successfully!


In [39]:
# Create logistic regression model
model = LogisticRegression(max_iter=1000)

# Define parameters to test
parameters = {
    "C": [0.01, 0.1, 1, 10, 100]
}

# Find the best parameters
grid_search = GridSearchCV(
    model,
    parameters,
    cv=5,
    scoring="accuracy"
)

# Train the model
grid_search.fit(X_train, y_train)

# Make predictions
y_pred = grid_search.predict(X_test)

# Calculate accuracy
from sklearn.metrics import accuracy_score

best_score = accuracy_score(y_test, y_pred)

print("Best parameters:", grid_search.best_params_)
print("Accuracy:", best_score)

Best parameters: {'C': 0.1}
Accuracy: 0.8333333333333334


In [40]:
print("Best accuracy score:", best_score)
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Best accuracy score: 0.8333333333333334
Confusion matrix:
[[53  8]
 [15 62]]
